
This notebook runs statistical tests to identify features associated with "wonky" survey respondent behavior.

**Methods:**
- **OLS Regression**: Effect size estimates, p-values, and Cohen's d
- **Logistic Regression**: Odds ratios for binary outcome interpretation

Both use **cluster-robust standard errors** to account for respondent-level clustering.

---

## Output Columns

| Column | Description |
|--------|-------------|
| `n_wonky` / `n_non_wonky` | Sample sizes for outcome groups |
| `wonky_mean` / `non_wonky_mean` | Feature prevalence among wonky vs non-wonky respondents |
| `ols_coefficient` | Mean difference in outcome for feature=1 vs feature=0 |
| `cohens_d` | Standardized effect size |
| `ols_interpretation` | Plain English: "Increases wonkiness by X%" |
| `ols_stars` | Significance stars: *** (p<0.001), ** (p<0.01), * (p<0.05) |
| `odds_ratio` | OR>1 = risk factor, OR<1 = protective |
| `lr_interpretation` | Plain English: "X% more/less likely" or "X.Xx times" |
| `logit_stars` | Significance stars for logistic regression |

### Setup

In [0]:
# Import libraries
import sys
import os
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'src'))

import numpy as np
import pandas as pd
import yaml

# Local module - simplified statistical tests
from eda.statistical_tests import (
    run_statistical_tests,
    get_summary_table,
)
from eda.visualizations import (
    create_breakdown_summary,
    create_breakdown_chart,
)

# Load configs (your own code)
with open('../configs/statistical_tests.yaml', 'r') as f:
    stats_config = yaml.safe_load(f)

with open('../configs/data_paths.yaml', 'r') as f:
    paths_config = yaml.safe_load(f)

pd.set_option('display.max_columns', None)
print("✓ Imports and configs loaded successfully")

In [0]:
# Set paths
notebook_path = os.getcwd()
repo_root = os.path.abspath(os.path.join(notebook_path, ".."))
misc_dir = os.path.join(repo_root, "misc")

# Configuration
OUTCOME_VAR = 'wonky_study_count'
USER_ID_VAR = 'respondentPk'
SIGNIFICANCE_LEVEL = 0.1

print(f"Output directory: {misc_dir}")

In [0]:
import mlflow
mlflow.autolog(disable=True)

### Load

In [0]:
# Load user data
user_df_input_path = os.path.join(
    misc_dir,
    os.path.basename(paths_config['output_files']['user_info_df_post_eda'])
)

user_info_df = pd.read_parquet(user_df_input_path)
print(f"Loaded data: {user_info_df.shape}")

In [0]:
# Filter to valid exposure data
user_info_df = user_info_df[~user_info_df['exposure_band'].isna()].reset_index(drop=True) # data that only exists in balance tables.
print(f"After filtering: {user_info_df.shape}")
print(f"\nOutcome distribution:")
print(f"  Mean {OUTCOME_VAR}: {user_info_df[OUTCOME_VAR].mean():.4f}")
print(f"  Wonky rate: {(user_info_df[OUTCOME_VAR] > 0).mean():.2%}")

In [0]:
9499 * 0.4243

### Statistical Testing

Run OLS and Logistic regression tests on all feature sets to identify significant predictors of wonkiness.

In [0]:
# Run tests for each feature set
results = []
feature_sets = stats_config["feature_sets"]

for feature_set_name in feature_sets:
    print("-" * 100)
    print(f"Working on {feature_set_name}")
    print("-" * 100)
    
    feature_list = stats_config["feature_sets"][feature_set_name]
    
    # Skip if feature list is None or empty
    if not feature_list:
        print(f"  Skipping - no features defined")
        continue
    
    # Run combined OLS + Logistic tests
    combined_df = run_statistical_tests(
        df=user_info_df,
        feature_list=feature_list,
        outcome_var=OUTCOME_VAR,
        user_id_var=USER_ID_VAR,
        significance_level=SIGNIFICANCE_LEVEL,
        n_jobs=-1,
        verbose=False,
    )
    
    if len(combined_df) > 0:
        combined_df["feature_set"] = feature_set_name
        combined_df = combined_df.reset_index()
        results.append(combined_df)
        print(f"{len(combined_df)} features tested")
    else:
        print(f"No valid features")

# Combine all results
results_df = pd.concat(results, ignore_index=True)
results_df = results_df.set_index(["feature_set", "feature"])

print(f"\n{'=' * 50}")
print(f"Total features tested: {len(results_df)}")

In [0]:
# Re-run the full test loop
results = []
feature_sets = stats_config["feature_sets"]

for feature_set_name in feature_sets:
    print("-" * 100)
    print(f"Working on {feature_set_name}")
    print("-" * 100)
    
    feature_list = stats_config["feature_sets"][feature_set_name]
    
    if not feature_list:
        print(f"  Skipping - no features defined")
        continue
    
    combined_df = run_statistical_tests(
        df=user_info_df,
        feature_list=feature_list,
        outcome_var=OUTCOME_VAR,
        user_id_var=USER_ID_VAR,
        significance_level=SIGNIFICANCE_LEVEL,
        n_jobs=-1,
        verbose=False,
    )
    
    if len(combined_df) > 0:
        combined_df["feature_set"] = feature_set_name
        combined_df = combined_df.reset_index()
        results.append(combined_df)
        print(f"  ✓ {len(combined_df)} features tested")
    else:
        print(f"  ⚠ No valid features")

results_df = pd.concat(results, ignore_index=True)
results_df = results_df.set_index(["feature_set", "feature"])

print(f"\n{'=' * 50}")
print(f"Total features tested: {len(results_df)}")

### Review

In [0]:
# High confidence results (significant in BOTH tests)
sig_both = results_df[results_df['significant_both'] == True]

print("=" * 70)
print("SIGNIFICANT IN BOTH OLS AND LOGISTIC (High Confidence)")
print("=" * 70)
print(f"Found {len(sig_both)} features significant in both tests")

display_cols = [
    'n_wonky', 'n_non_wonky',
    'wonky_mean', 'non_wonky_mean',
    'ols_coefficient', 'ols_p_value', 'ols_stars', 'ols_interpretation',
    'odds_ratio', 'logit_p_value', 'logit_stars', 'lr_interpretation'
]
available_cols = [c for c in display_cols if c in sig_both.columns]
display(sig_both[available_cols].reset_index())

In [0]:
# Summary table
summary = get_summary_table(results_df, top_n=25)
display(summary)

In [0]:
cols_to_remove = ['ols_se', 'cohens_d', 'or_ci_lower', 'or_ci_upper', 'logit_se', 'logit_z_stat']

In [0]:
# reduced view
results_df.drop(columns=cols_to_remove).reset_index().display()

### Export

In [0]:
cols_to_print = [
    "feature_set",
    "feature",
    "feature_type",
    "n_with_feature",
    "n_without_feature",
    "mean_outcome_with_feature",
    "mean_outcome_without_feature",
    "ols_coefficient",
    "ols_interpretation",
    "ols_p_value",
    "odds_ratio",
    "lr_interpretation",
    "logit_p_value",
    "ols_stars",
    "logit_stars",
    "significant_both"
] + ['ols_significant', 'ols_interpretation_short', 'cohens_d', 'cohens_d_magnitude', 'lr_interpretation_short']

results_df = results_df.reset_index()[cols_to_print]

In [0]:
# Export results for use in modelling notebook
test_results_path = os.path.join(
    misc_dir,
    os.path.basename(paths_config['output_files'].get('test_results_df'))
)

results_df.reset_index().to_csv(test_results_path, index=False)
print(f"✓ Results exported to: {test_results_path}")

In [0]:
results_df

In [0]:
results_df.reset_index()['feature'].value_counts()

### For individual Charting

Use this section to chart deltas of wonky vs non wonky by feature_sets

In [0]:
stats_config["feature_sets"]

In [0]:
# Define the set_name here from dictionary aboce
set_name = "days_active_exclusive"

In [0]:
item = results_df.reset_index()

binary_item = item[~item['mean_outcome_with_feature'].isna()]

feature_list = binary_item[binary_item['feature_set'] == set_name]['feature'].tolist()

In [0]:
# proportions break down and differences
print(
    create_breakdown_summary(
        user_info_df, # user level dataframe 
        features=feature_list, # features to test
        group_col="wonky_study_count", # outcome: should be binary outcome
    )
)

In [0]:
# all_segments = [
#     "days_active_0_to_2",
#     "days_active_3_to_10",
#     "days_active_11_to_20",
#     "days_active_21_to_30",
#     "days_active_31_to_50",
#     "days_active_51_to_75",
#     "days_active_76_to_100",
#     "days_active_101_to_125",
#     "days_active_126_to_150",
#     "days_active_151_to_200",
#     "days_active_201_to_250",
#     "days_active_251_plus",
# ]

# wonky_ones = [
#     "days_active_0_to_2",
#     "days_active_3_to_10",
#     "days_active_11_to_20",
#     "days_active_21_to_30",
#     "days_active_51_to_75",
#     "days_active_76_to_100",
#     "days_active_101_to_125",
#     "days_active_126_to_150",
# ]

How do the proportions of wonkiner catagories change when you compare non wonky groups to wonky groups?

In [0]:
# # Non wonky group
# all_segment_num = user_info_df[user_info_df["wonky_study_count"] == 0][all_segments].sum().sum()
# print(f"ALL SEGMENTS {all_segment_num}")
# wonky_ones_num = user_info_df[user_info_df["wonky_study_count"] == 0][wonky_ones].sum().sum()
# print(f"WONKY SEGEMENTS: {wonky_ones_num}")


# print(f"Wonky {wonky_ones_num/all_segment_num} proporitoned when only looking at non wonky")

In [0]:
# # Wonky group
# all_segment_num = user_info_df[user_info_df["wonky_study_count"] > 0][all_segments].sum().sum()
# print(f"ALL SEGMENTS {all_segment_num}")
# wonky_ones_num = user_info_df[user_info_df["wonky_study_count"] > 0][wonky_ones].sum().sum()
# print(f"WONKY SEGEMENTS: {wonky_ones_num}")


# print(f"Wonky {wonky_ones_num/all_segment_num} proporitoned when only looking at wonky")

Although total population in non wonky group is higher (5559) than the wonky group (3940). Count of grouped wonky segments (wonky days) is higher in the wonky group than in the non wonky group.

In [0]:
feature_list = ['days_active_0_to_2',
 'days_active_3_to_10',
 'days_active_11_to_20',
 'days_active_21_to_30',
 'days_active_31_to_50',
 'days_active_51_to_75',
 'days_active_76_to_100',
 'days_active_101_to_125',
 'days_active_126_to_150',
 'days_active_151_to_200',
 'days_active_201_to_250',
 'days_active_251_plus']

In [0]:
# proportions break down and differences
create_breakdown_chart(
    user_info_df, # user level dataframe 
    features=feature_list, # features to test
    group_col="wonky_study_count", # outcome: should be binary outcome
)